# Closure DeepONet: two splitting protocols and uncertainty quantification**Pipeline.** CO2 GCS dataset -> preprocessing -> physics-based feature engineering ->case-level splitting -> feature scaling -> Closure DeepONet -> MC-Dropout / Deep Ensemble / Bayesian -> predictive samples -> mean + uncertainty -> calibration -> metrics -> decision support.**Architecture.** Softmax over four logits enforces closure and non-negativity exactly, with no penalty weight to tune.All modelling code lives in the `co2gcs` package so that the four notebooks shareone implementation. Put `co2gcs/` and `All_dataset.csv` next to this notebook.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerimport co2gcs as CC.plots.use_style()SEED = 42TARGETS = C.TARGETS

## 1. Data and physics-based feature engineering

In [ ]:
# ---- load + physics-based feature engineering -------------------------------# fix_injection=True restores each case's injection rate over the full trajectory.# In the raw file the column is zeroed for every t > 30 yr, which leaks the# post-injection regime label and deletes the strongest control on long-term# trapping across 85% of the horizon.df = C.load('All_dataset.csv', fix_injection=True)Xb, Xt, y, names_b, names_t = C.build_features(df, mode='fixed')groups = df['case'].valuestvals  = df['Time (years)'].valuesprint(f'{len(df)} rows, {df.case.nunique()} simulation cases, '      f'branch dim {Xb.shape[1]}, trunk dim {Xt.shape[1]}')df.head()

## 2. Splitting protocols

In [ ]:
# ---- the two splitting protocols --------------------------------------------PROTOCOLS = ['random', 'group']def get_split(protocol, fold=0):    tr, va, te = C.make_split(protocol, len(y), groups, seed=SEED, n_folds=5, fold=fold)    sb = StandardScaler().fit(Xb[tr])      # scalers fit on TRAIN ONLY    st = StandardScaler().fit(Xt[tr])    pack = lambda m: (sb.transform(Xb[m]).astype('float32'),                      st.transform(Xt[m]).astype('float32'), y[m])    d = {}    for nm, m in [('tr', tr), ('va', va), ('te', te)]:        d[f'Xb_{nm}'], d[f'Xt_{nm}'], d[f'y_{nm}'] = pack(m)    d.update(t_te=tvals[te], g_te=groups[te], mask_te=te,             dims=(Xb.shape[1], Xt.shape[1]), scalers=(sb, st))    return dfor p in PROTOCOLS:    d = get_split(p)    print(f"{p:7s} train {len(d['y_tr']):6d}  val {len(d['y_va']):5d}  "          f"test {len(d['y_te']):5d}  test cases {len(np.unique(d['g_te'])):4d}")

### Pros and cons of each protocol

In [ ]:
for p in PROTOCOLS:    n = C.PROTOCOL_NOTES[p]    print('=' * 78); print(n['name']); print('=' * 78)    print(' PROS'); [print('   +', x) for x in n['pros']]    print(' CONS'); [print('   -', x) for x in n['cons']]    print()

## 3. Train and compare the protocols

In [ ]:
# ---- train one model per protocol -------------------------------------------db, dt = Xb.shape[1], Xt.shape[1]TRAIN_KW = dict(epochs=80, patience=30, bs=1024, lr=1.5e-3, wd=1e-5, var_weight=True)results, preds, splits = [], {}, {}for p in PROTOCOLS:    d = get_split(p); splits[p] = d    C.seed_all(SEED)    drop, head, bayes = 0.0, 'point', False    model = C.ClosureDeepONet(db, dt, sigma=0.3, p_drop=drop, head=head, bayes_last=bayes)    model, hist = C.train_model(model, d['Xb_tr'], d['Xt_tr'], d['y_tr'],                                d['Xb_va'], d['Xt_va'], d['y_va'],                                head='point', **TRAIN_KW)    P = C.predict_mean(model, d['Xb_te'], d['Xt_te'])    preds[p] = P    m = C.accuracy(d['y_te'], P); m['protocol'] = p    results.append(m)    print(f"{p:7s} R2_mean={m['R2_mean']:.4f}  " +          '  '.join(f'{t}={m[f"R2_{t}"]:.3f}' for t in TARGETS) +          f"  closureMAE={m['closure_MAE']:.4f}")res = pd.DataFrame(results)[['protocol'] + [f'R2_{t}' for t in TARGETS] +                            ['R2_mean'] + [f'RMSE_{t}' for t in TARGETS] +                            ['MAE_mean', 'closure_MAE']]res.round(4)

### GroupKFold spread

In [ ]:
# ---- 5-fold GroupKFold: fold-to-fold spread is the deployment variance -------fold_rows = []for f in range(5):    d = get_split('group', fold=f)    C.seed_all(SEED + f)    drop, head, bayes = 0.0, 'point', False    model = C.ClosureDeepONet(db, dt, sigma=0.3, p_drop=drop, head=head, bayes_last=bayes)    model, _ = C.train_model(model, d['Xb_tr'], d['Xt_tr'], d['y_tr'],                             d['Xb_va'], d['Xt_va'], d['y_va'],                             head='point', **TRAIN_KW)    m = C.accuracy(d['y_te'], C.predict_mean(model, d['Xb_te'], d['Xt_te']))    m['fold'] = f; fold_rows.append(m)    print(f"fold {f}: R2_mean={m['R2_mean']:.4f}")folds = pd.DataFrame(fold_rows)print('\nGroupKFold R2_mean = %.4f +/- %.4f' % (folds.R2_mean.mean(), folds.R2_mean.std()))

## 4. Performance figures

In [ ]:
# ---- figures ----------------------------------------------------------------import os; os.makedirs('figs', exist_ok=True)C.plots.fig_protocol_bars(res, 'figs/closure_protocol_bars.png',                          title='Closure DeepONet: accuracy under both protocols')C.plots.fig_parity({{'Protocol 1: random rows': splits['random']['y_te'],                    'Protocol 2: unseen cases': splits['group']['y_te']}},                   {{'Protocol 1: random rows': preds['random'],                    'Protocol 2: unseen cases': preds['group']}},                   'figs/closure_parity.png', title='Closure DeepONet: parity by protocol')C.plots.fig_trajectories(splits, preds, 'figs/closure_trajectories.png', n=5,                         title='Closure DeepONet: five randomly drawn simulations')for f in ['closure_protocol_bars', 'closure_parity', 'closure_trajectories']:    display_png = f'figs/{{f}}.png'    print(display_png)

In [ ]:
from IPython.display import Image, displayfor f in ['closure_protocol_bars', 'closure_parity', 'closure_trajectories']:    display(Image(f'figs/{{f}}.png'))

## Uncertainty quantificationThree engines are implemented and selectable through `METHOD`:| method | cost | epistemic source | notes ||---|---|---|---|| `mc_dropout` | 1 training run + S passes | dropout masks | cheapest; the dropout rate is an unprincipled knob and it tends to be over-dispersed in a way that flatters PICP while worsening ECE || `deep_ensemble` | M training runs | independent optima | strongest and best calibrated after recalibration; recommended default || `bayesian` | 1 training run | variational posterior on the final projection | principled and cheap. Natural for a DeepONet because the output is *linear in the learned basis*, so the last layer carries most of the predictive uncertainty. Full VI over every weight costs far more and does not generally beat ensembles |`deterministic` is included as an accuracy reference only; PICP and ECE aredegenerate for it because it has no predictive spread.**Aleatoric vs. epistemic.** The head supplies aleatoric noise, the enginesupplies epistemic spread, and the two combine by the law of total variance(`C.decompose`).Aleatoric noise is a **logit-normal (additive-log-ratio)** head: the network emits the mean composition plus a per-coordinate log-sigma in ALR space, sampling perturbs the ALR coordinates, and the softmax maps back. Every predictive sample is therefore a valid composition summing to 1.A Dirichlet head was tried first and fails on this data: it ties all four components to a single concentration parameter, and because MTI is ~3 orders of magnitude smaller than STTI the concentration pins to its floor and the mean collapses (R2 = -3.95). The ALR head is scale-free per component and does not have this failure mode.**Calibration is not optional.** Ensembles and the Bayesian layer come outunder-dispersed on unseen cases; a single post-hoc scale factor per index,fitted on the validation partition, brings observed coverage back to nominalwithout touching the predictive mean.

In [ ]:
RENORM = True   # keep recalibrated samples on the simplex

In [ ]:
# ---- select the UQ engine ---------------------------------------------------METHOD    = 'deep_ensemble'     # 'mc_dropout' | 'deep_ensemble' | 'bayesian' | 'deterministic'N_SAMPLES = 200                 # stochastic forward passes (100-200 recommended)N_MEMBERS = 5                   # ensemble membersLEVEL     = 0.90                # nominal interval levelHEAD      = 'logitnormal'P_DROP    = 0.10def build(method):    drop  = P_DROP if method == 'mc_dropout' else 0.0    bayes = (method == 'bayesian')    head  = HEAD if method != 'deterministic' else 'point'    db, dt = Xb.shape[1], Xt.shape[1]    fn = lambda s: C.ClosureDeepONet(db, dt, sigma=0.3, p_drop=drop, head=head, bayes_last=bayes)    return fn, headdef run_one(method, protocol):    d  = get_split(protocol)    fn, head = build(method)    tk = dict(TRAIN_KW); tk['head'] = head    if method == 'bayesian':        tk['kl_weight'] = 0.05          # tempered: the MSE-scaled likelihood is                                        # otherwise swamped by the KL term    r = C.run_uq(method, fn, d, head=head, n_samples=N_SAMPLES,                 n_members=N_MEMBERS, seed=SEED, train_kw=tk)    return d, r, head

### Run every engine under both protocols

In [ ]:
# ---- run every engine under both protocols ----------------------------------METHODS = ['deterministic', 'mc_dropout', 'deep_ensemble', 'bayesian']rows, store = [], {}for protocol in PROTOCOLS:    for method in METHODS:        d, r, head = run_one(method, protocol)        S = r['samples']        if method != 'deterministic':            # post-hoc variance recalibration, fitted on VALIDATION only            from co2gcs.uq import _forward_all, _enable_dropout            rng = np.random.default_rng(SEED); Sv = []            for m in r['models']:                if method == 'mc_dropout':                    _enable_dropout(m)                for _ in range(10):                    Sv.append(_forward_all(m, d['Xb_va'], d['Xt_va'], head, rng,                                           bayes_sample=(method == 'bayesian')))                m.eval()            scale = C.fit_variance_scaling(d['y_va'], np.stack(Sv))            S_cal = C.apply_variance_scaling(S, scale, renormalise=RENORM)        else:            scale, S_cal = np.ones(4), S        met,  cc  = C.summarise(d['y_te'], S,     level=LEVEL)        metc, ccc = C.summarise(d['y_te'], S_cal, level=LEVEL)        rows.append(dict(protocol=protocol, method=method, **met,                         **{f'cal_{k}': v for k, v in metc.items()}))        store[(protocol, method)] = dict(S=S, S_cal=S_cal, cc=cc, cc_cal=ccc,                                         d=d, scale=scale)        print(f'{protocol:7s} {method:14s} R2={met["R2_mean"]:.4f} '              f'PICP={met["PICP_mean"]:.3f}->{metc["PICP_mean"]:.3f} '              f'NMPIW={metc["NMPIW_mean"]:.3f} CRPS={met["CRPS_mean"]:.4f} '              f'ECE={met["ECE_mean"]:.3f}->{metc["ECE_mean"]:.3f} '              f'closure={met["closure_MAE"]:.4f}')uq = pd.DataFrame(rows)uq[['protocol', 'method', 'R2_mean', 'RMSE_mean', 'PICP_mean', 'cal_PICP_mean',    'cal_NMPIW_mean', 'CRPS_mean', 'ECE_mean', 'cal_ECE_mean', 'closure_MAE']].round(4)

### Uncertainty figures

In [ ]:
# ---- UQ figures -------------------------------------------------------------os.makedirs('figs', exist_ok=True)best = 'deep_ensemble'C.plots.fig_calibration({m: store[('group', m)]['cc'] for m in METHODS[1:]},                        'figs/closure_reliability_raw.png',                        title='Closure DeepONet: reliability before recalibration (unseen cases)')C.plots.fig_calibration({m: store[('group', m)]['cc_cal'] for m in METHODS[1:]},                        'figs/closure_reliability_cal.png',                        title='Closure DeepONet: reliability after recalibration (unseen cases)')tbl = uq[uq.method != 'deterministic'].copy()tbl['PICP_mean'] = tbl['cal_PICP_mean']; tbl['NMPIW_mean'] = tbl['cal_NMPIW_mean']C.plots.fig_picp_mpiw(tbl, 'figs/closure_picp_mpiw.png',                      title='Closure DeepONet: coverage, sharpness and CRPS')st = store[('group', best)]rng = np.random.RandomState(7)cases = rng.choice(np.unique(st['d']['g_te']), 5, replace=False)C.plots.fig_uncertainty_bands(st['d']['y_te'], st['S_cal'], st['d']['t_te'],                              st['d']['g_te'], cases, 'figs/closure_bands.png', level=LEVEL,                              title='Closure DeepONet: 90% intervals on unseen simulations')dec = C.decompose(np.array_split(st['S_cal'], N_MEMBERS))C.plots.fig_uncertainty_vs_time(st['d']['y_te'], st['S_cal'], st['d']['t_te'],                                'figs/closure_uncertainty_time.png', decomposition=dec,                                title='Closure DeepONet: uncertainty vs. storage horizon')C.plots.fig_risk(st['d']['y_te'], st['S_cal'], 'figs/closure_risk.png', index='RTI',                 title='Closure DeepONet: risk-informed screening on unseen sites')print('figures written to figs/')

In [ ]:
from IPython.display import Image, displayfor f in ['closure_reliability_raw', 'closure_reliability_cal', 'closure_picp_mpiw',          'closure_bands', 'closure_uncertainty_time', 'closure_risk']:    display(Image(f'figs/{{f}}.png'))

### 5. Reading the results* Compare `PICP_mean` with `cal_PICP_mean`. Raw PICP is a poor ranking statistic:  MC dropout can land near nominal simply by being over-dispersed while having  the worst ECE of the three engines. Rank on **ECE and CRPS**, and report PICP  and MPIW together - coverage without sharpness is meaningless, since an  arbitrarily wide interval attains any PICP.* `closure_MAE` is exactly 0 for the closure architecture and non-zero for the  baseline, including for the *samples*, not merely the mean.* The protocol gap appears in the probabilistic metrics too: CRPS and raw ECE  are both worse under grouped splitting. Random splitting inflates apparent  calibration quality as well as apparent accuracy.